# COMP3122 — Week 2 Demo Notebook
### Chapter 2: NumPy — Arrays and Operations

**Applied Data Science**

This notebook is the live-coding companion to the Week 2 lecture. It follows the same five
blocks as the slides, in the same order, with the same examples.

Type the cells in as we go. Copying is faster; typing is how the patterns stick.

---
## Setup

One line, at the top of every notebook from now on.

In [ ]:
import numpy as np    # explain: the standard short name for the library

print("NumPy version:", np.__version__)   # explain: worth knowing which version you are on

---
## Block 1 — Why arrays are fast  ·  Book §2.1

A Python list can hold anything, so every item carries its own type tag and Python checks
that tag at every operation. A NumPy array holds one type only, with the values side by side
in memory, and the work runs in compiled C.

Let us measure it rather than assert it.

In [ ]:
import time

np.sum(np.arange(10) ** 2)      # warm-up, so the first timing is not misleading

start = time.perf_counter()
python_result = sum([x**2 for x in range(1_000_000)])
python_ms = (time.perf_counter() - start) * 1000

start = time.perf_counter()
arr = np.arange(1_000_000)
numpy_result = np.sum(arr ** 2)
numpy_ms = (time.perf_counter() - start) * 1000

print("python:", python_result, f"in {python_ms:.1f} ms")
print("numpy :", numpy_result, f"in {numpy_ms:.1f} ms")
print("same answer:", python_result == numpy_result)
print(f"ratio: {python_ms / numpy_ms:.0f}x")

The first run is the slowest — the ratio settles after a warm-up. Run the cell two or three
times and watch it climb. In a notebook you would normally use `%timeit`, which does the
repeats for you:

In [ ]:
%timeit sum([x**2 for x in range(1_000_000)])   # explain: %timeit repeats it and reports the mean and spread
%timeit np.sum(np.arange(1_000_000) ** 2)       # explain: same answer, many times faster

Expect a big ratio — twenty times, forty, sixty. It depends on the machine, the Python
version and the NumPy version, so the number on your screen will not match anyone else's.
The size of the gap is the point, not the exact figure.

---
## Block 2 — Making and reading arrays  ·  Book §2.2

An array is a row of mailboxes: same size, side by side, one number each, opened by number
starting at zero.

In [ ]:
monday_temps = np.array([18, 21, 23, 22, 19])   # explain: five readings, one per time of day
print(monday_temps)                              # explain: no commas in the output -- that is NumPy, not a list

No commas in the printed output. That is NumPy reminding you it is not a list.

### Five ways to make an array

In [ ]:
print(np.array([18, 21, 23]))     # explain: from a list you already have
print(np.zeros(5))                # explain: five zeros   (floats)
print(np.ones(3))                 # explain: three ones   (floats)
print(np.arange(0, 10, 2))        # explain: takes a STEP -- steps of two, stops BEFORE 10
print(np.linspace(0, 1, 5))       # explain: takes a COUNT -- five points, and 1 IS included

`arange` takes a **step**. `linspace` takes a **count**.

Avoid `arange` with fractional steps — floating-point rounding makes the last value
unpredictable. Use `linspace` instead:

In [ ]:
print(np.arange(0, 1, 0.1))       # explain: fractional steps drift -- looks fine here, but do not rely on it
print(np.linspace(0, 0.9, 10))    # explain: say what you want: ten points

### The four attributes you always check

No parentheses. These are attributes, not methods.

In [ ]:
print(monday_temps.shape)    # explain: (5,)     five values along one axis
print(monday_temps.ndim)     # explain: 1        one axis
print(monday_temps.size)     # explain: 5        five values in total
print(monday_temps.dtype)    # explain: int64    the type every value shares

The lonely comma in `(5,)` is what makes it a tuple rather than the number five.

`ndim` counts **axes**. `size` counts **values**. A 1-D array of a million numbers still
has `ndim` of 1.

Now watch the classic first-day error:

In [ ]:
try:
    monday_temps.shape()       # explain: shape is an ATTRIBUTE -- calling it is the error
except TypeError as e:
    print("TypeError:", e)     # explain: meet this message here, on purpose

### Our running example — four cities, seven days

Rows are cities, columns are days. This array is the rest of the chapter, and it comes back
in Chapter 3.

In [ ]:
temps = np.array([
    [ 8, 10, 12, 11,  9, 13, 14],   # Toronto [0]
    [22, 23, 21, 24, 25, 26, 24],   # Cairo   [1]
    [15, 16, 14, 17, 18, 16, 17],   # Tokyo   [2]
    [20, 22, 21, 23, 22, 21, 20],   # Sydney  [3]
])
# columns: Mon [0]  Tue [1]  Wed [2]  Thu [3]  Fri [4]  Sat [5]  Sun [6]

print(temps)
print("shape:", temps.shape, " ndim:", temps.ndim, " size:", temps.size)

---
## Block 3 — dtype: why it matters  ·  Book §2.3

Every value in an array shares one type. If any value is a decimal, they all become decimals.

In [ ]:
print(np.array([1, 2, 3]).dtype)        # explain: int64    all whole
print(np.array([1, 2, 3.0]).dtype)      # explain: float64  one decimal wins
print(np.array([0, 1, 0, 1], dtype=bool))   # explain: force a type yourself with dtype=

What if you mix in a word? Worth seeing once, and never doing on purpose.

In [ ]:
mixed = np.array([1, 2, "three"])   # explain: one type for the whole array, so something has to give
print(mixed, mixed.dtype)     # explain: everything became text — arithmetic now fails

### Trap one — integer division

Predict the last line before you run it.

In [ ]:
counts = np.array([10, 20, 30])

print(counts / 2,  (counts / 2).dtype)     # explain: regular division -> floats
print(counts // 2, (counts // 2).dtype)    # explain: floor division -> rounds DOWN, stays whole

print(np.array([1, 2, 3]) // 2)            # explain: rounded down -- 1//2 is 0, 3//2 is 1

That result is not a bug. It is the operator: `/` is true division and gives floats; `//` is floor
division and rounds every quotient down (for floats too — `np.array([1., 2., 3.]) // 2` is `[0., 1., 1.]`).
The dtype only decides how the result is stored.

### Trap two — `/=` on an integer array

In [ ]:
arr = np.array([1, 2, 3])
arr = arr / 2          # explain: fine: builds a NEW float array
print(arr, arr.dtype)  # explain: the dtype changed because the array did

arr = np.array([1, 2, 3])
try:
    arr /= 2           # explain: in place, and the answer needs decimals
except Exception as e:
    print(type(e).__name__)   # explain: UFuncTypeError -- floats will not fit in an int array
    print(e)

NumPy will not let an in-place operation change the array's dtype. The same happens with
`arr *= 1.5` and `arr += 1.5` on an integer array.

Two fixes: build it as float from the start, or stop using `/=`.

In [ ]:
arr = np.array([1, 2, 3], dtype=float)   # explain: say float up front
arr /= 2                                 # explain: now in-place division is fine
print(arr)

---
## Block 4 — Indexing and slicing  ·  Book §2.4

### One dimension

In [ ]:
toronto = np.array([8, 10, 12, 11, 9, 13, 14])    # Toronto [0]

print(toronto[0])      # explain: counting starts at zero -- 8    Monday [0]
print(toronto[-1])     # explain: 14   Sunday [6], counting from the end

print(toronto[0:3])    # explain: Mon [0], Tue [1], Wed [2]   — stop is EXCLUDED
print(toronto[2:5])    # explain: three values again, shifted along
print(toronto[4:])     # explain: from index 4 to the end

Adding a step. The full form is `start:stop:step`, and every part has a default.

In [ ]:
print(toronto[::2])     # explain: the third number is the STEP -- from the start, every second value
print(toronto[1::2])    # explain: from index 1
print(toronto[3::2])    # explain: from index 3
print(toronto[::-1])    # explain: a useful trick: reversed

### Two dimensions — one bracket, row then column

In [ ]:
print(temps[0, 0])     # explain: row first, then column -- Toronto [0], Mon [0]
print(temps[1, 3])     # explain: Cairo   [1], Thu [3]
print(temps[3, -1])    # explain: negatives work per axis -- Sydney  [3], Sun [6]

`temps[1][3]` gives the same answer. Do not use it — the comma form is faster, clearer, and
lets you slice both axes together, which is next.

In [ ]:
print(temps[0, :])       # explain: Toronto's [0] whole week   — : means all columns
print(temps[:, 0])       # explain: Monday [0] in every city   — : means all rows
print(temps[1:3, 0:4])   # explain: slice both axes at once -- Cairo [1] and Tokyo [2], Mon [0] to Thu [3]

### The trap — a slice is a view

Predict what the third line prints before you run this.

In [ ]:
monday = temps[:, 0]     # explain: a slice is a VIEW -- it shares memory with temps
print(monday)

monday[0] = 999          # explain: writing here writes there
print(temps[0, 0])       # explain: the original changed too

temps[0, 0] = 8          # explain: put it back

In [ ]:
monday = temps[:, 0].copy()      # explain: ask for a real copy
monday[0] = 999                  # explain: this now writes nowhere near temps
print(temps[0, 0])               # explain: untouched this time
print(temps[:, 0])

A slice is a window into the same data, not a photograph of it. Ask yourself: do I want my
change to show up in the original? If not, add `.copy()`.

---
## Block 5 — Universal functions  ·  Book §2.5

One expression, applied to every element, with no loop. That is vectorization.

In [ ]:
toronto = temps[0, :]

print(toronto + 5)      # explain: no loop -- a warmer week
print(toronto * 2)      # explain: every value doubled
print(toronto ** 2)     # explain: every value squared
print(toronto)          # explain: unchanged — the operations returned NEW arrays

That last line matters. `toronto + 5` does not modify `toronto`. To keep the result, assign it.

### Two arrays, element by element

In [ ]:
monday_all  = temps[:, 0]     # explain: Monday [0], four cities
tuesday_all = temps[:, 1]     # explain: Tuesday [1], four cities

print(monday_all)
print(tuesday_all)
print(tuesday_all - monday_all)     # explain: same shape, so it subtracts position by position -- how much warmer Tuesday was, per city

Same shape in, same shape out. Position 0 pairs with position 0.

### Useful whole-array functions

In [ ]:
print(np.sqrt(np.array([1, 4, 9, 16, 25])))     # explain: a ufunc -- side lengths of those squares
print(np.abs(np.array([-3, -1, 0, 2, 5])))      # explain: distance from zero
print(np.round(np.array([1.4, 2.5, 3.6, 4.5])))  # explain: look carefully at 2.5 and 4.5
print(np.maximum(np.array([1, 2, 3]), np.array([4, 5, 6])))   # explain: element-by-element max of two arrays

`2.5` rounded to `2` and `4.5` to `4`. Exact halves go to the nearest **even** number. That is
a real rule, not a bug — check the output whenever halves matter.

`sqrt`, `abs` and `maximum` are ufuncs. `round` behaves like one but NumPy does not classify
it as a ufunc, and neither is `sum`.

### The big payoff

In [ ]:
arr = np.arange(1_000_000)   # explain: a million values
print(np.sum(arr ** 2))      # explain: square them all and sum, in one line

---
## The mindset shift  ·  Book §2.6

Three loops from Chapter 1, and their one-line array versions.

In [ ]:
numbers = np.array([4, 8, 15, 16, 23, 42])

# Before
# explain: build an empty list, append on every pass
result = []
for x in numbers:
    result.append(x + 5)
print(result)

# After
print(numbers + 5)   # explain: four lines become one

In [ ]:
# Before
squares = [x ** 2 for x in numbers]
print(squares)

# After
print(numbers ** 2)   # explain: shorter still, and it runs in C

In [ ]:
a = np.array([1, 2, 3])
b = np.array([10, 20, 30])

# Before
print([x * y for x, y in zip(a, b)])

# After
print(a * b)   # explain: same shape in, same shape out

**The rule.** If you are writing a Python `for` loop over the elements of an array, stop.
There is almost certainly a one-line array version — shorter, faster, and closer to the
sentence you would say out loud.

---
## Before next class

Review Chapter 2 · finish Lab 02 · read Chapter 3.

Chapter 3 asks these same arrays questions — the average, the highest, the coldest day — and
introduces broadcasting. The four cities come back, so make sure you can build `temps` from
memory.